In [8]:
import sys
import warnings
sys.path.append('..')

import pandas as pd
import numpy as np
from sklearn.linear_model import Lasso, Ridge
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import VotingRegressor, StackingRegressor

from src.config import TRAIN_PATH, TEST_PATH, RANDOM_SEED, CV_N_SPLITS, SUBMISSIONS_DIR
from src.preprocessing import HousePricesPreprocessor
from src.features import add_features
from src.validation import cross_validate

warnings.filterwarnings('ignore')

In [9]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

In [10]:
y = np.log1p(train['SalePrice'])

# Выбросы
outlier_idx = train[(train['GrLivArea'] > 4000) & (train['SalePrice'] < 200000)].index
train = train.drop(outlier_idx)
y = y.drop(outlier_idx)

test_ids = test['Id']
train = train.drop(['SalePrice', 'Id'], axis=1)
test = test.drop('Id', axis=1)

# Preprocessing → Feature Engineering
prep_raw = HousePricesPreprocessor(scale=False)
X_raw = prep_raw.fit_transform(train, np.expm1(y))
X_test_raw = prep_raw.transform(test)
X_raw = add_features(X_raw)
X_test_raw = add_features(X_test_raw)

prep_scaled = HousePricesPreprocessor(scale=True)
X_scaled = prep_scaled.fit_transform(train, np.expm1(y))
X_test_scaled = prep_scaled.transform(test)
X_scaled = add_features(X_scaled)
X_test_scaled = add_features(X_test_scaled)

print(f"X_raw: {X_raw.shape}, X_scaled: {X_scaled.shape}")
print(f"NaN raw: {X_raw.isnull().sum().sum()}, NaN scaled: {X_scaled.isnull().sum().sum()}")

X_raw: (1458, 107), X_scaled: (1458, 107)
NaN raw: 0, NaN scaled: 0


In [11]:
models = {
    'Lasso': (Lasso(alpha=0.005), X_scaled, y),
    'Ridge': (Ridge(alpha=100), X_scaled, y),
    'RF': (RandomForestRegressor(n_estimators=500, max_depth=15, random_state=RANDOM_SEED), X_raw, y),
    'CatBoost': (CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6, random_seed=RANDOM_SEED, verbose=0), X_raw, y),
    'XGBoost': (XGBRegressor(n_estimators=500, max_depth=3, learning_rate=0.05, random_seed=RANDOM_SEED, verbosity=0), X_raw, y),
    'LightGBM': (LGBMRegressor(n_estimators=500, max_depth=4, learning_rate=0.05, random_seed=RANDOM_SEED, verbose=-1), X_raw, y),
}

results = {}
for name, (model, X_data, y_data) in models.items():
    print(f"\n=== {name} ===")
    scores = cross_validate(model, X_data, y_data)
    results[name] = scores.mean()


=== Lasso ===
RMSE по фолдам: [0.1101 0.1294 0.107  0.1175 0.1153]
Среднее: 0.1159 ± 0.0077

=== Ridge ===
RMSE по фолдам: [0.1111 0.1287 0.1085 0.1154 0.1199]
Среднее: 0.1167 ± 0.0072

=== RF ===
RMSE по фолдам: [0.1261 0.1435 0.1275 0.1456 0.1288]
Среднее: 0.1343 ± 0.0084

=== CatBoost ===
RMSE по фолдам: [0.1062 0.1259 0.1038 0.12   0.1176]
Среднее: 0.1147 ± 0.0084

=== XGBoost ===
RMSE по фолдам: [0.113  0.1382 0.1077 0.133  0.1168]
Среднее: 0.1218 ± 0.0118

=== LightGBM ===
RMSE по фолдам: [0.1115 0.1357 0.112  0.1319 0.1255]
Среднее: 0.1233 ± 0.0100


VotingRegressor

In [12]:
voting = VotingRegressor(estimators=[
    ('lasso', Lasso(alpha=0.005)),
    ('catboost', CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6, random_seed=RANDOM_SEED, verbose=0)),
    ('xgboost', XGBRegressor(n_estimators=500, max_depth=3, learning_rate=0.05, random_seed=RANDOM_SEED, verbosity=0)),
])
scores_voting = cross_validate(voting, X_scaled, y)

RMSE по фолдам: [0.1055 0.1259 0.1013 0.1191 0.1112]
Среднее: 0.1126 ± 0.0089


StackingRegressor

In [13]:
stacking = StackingRegressor(
    estimators=[
        ('lasso', Lasso(alpha=0.005)),
        ('catboost', CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6, random_seed=RANDOM_SEED, verbose=0)),
        ('xgboost', XGBRegressor(n_estimators=500, max_depth=3, learning_rate=0.05, random_seed=RANDOM_SEED, verbosity=0)),
    ],
    final_estimator=Ridge(alpha=1.0),
    cv=CV_N_SPLITS,
)
scores_stacking = cross_validate(stacking, X_scaled, y)

RMSE по фолдам: [0.104  0.1253 0.1011 0.1175 0.1098]
Среднее: 0.1115 ± 0.0089


## Итоги

Одиночные модели (с FE):
- CatBoost (depth=6, lr=0.05):              CV=0.1147
- Lasso (alpha=0.005):                      CV=0.1159
- Ridge (alpha=100):                        CV=0.1167
- XGBoost (n_est=500, depth=3, lr=0.05):    CV=0.1218
- LightGBM (n_est=500, depth=4, lr=0.05):   CV=0.1233
- RF (500, depth=15):                        CV=0.1343

Ансамбли (Lasso + CatBoost + XGBoost):
- StackingRegressor (final=Ridge):          CV=0.1115  Kaggle=0.12417
- VotingRegressor:                          CV=0.1126  Kaggle=0.12341
- CatBoost (одиночный):                     CV=0.1147  Kaggle=0.12340 ← лучший на Kaggle

In [14]:
# 1. Лучшая одиночная модель — CatBoost
cat = CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6, random_seed=RANDOM_SEED, verbose=0)
cat.fit(X_raw, y)
preds_cat = np.expm1(cat.predict(X_test_raw)).clip(min=0)
sub_cat = pd.DataFrame({'Id': test_ids, 'SalePrice': preds_cat})
sub_cat.to_csv(SUBMISSIONS_DIR / 'catboost_fe.csv', index=False)

# 2. VotingRegressor
voting.fit(X_scaled, y)
preds_voting = np.expm1(voting.predict(X_test_scaled)).clip(min=0)
sub_voting = pd.DataFrame({'Id': test_ids, 'SalePrice': preds_voting})
sub_voting.to_csv(SUBMISSIONS_DIR / 'voting_top3.csv', index=False)

# 3. StackingRegressor
stacking.fit(X_scaled, y)
preds_stacking = np.expm1(stacking.predict(X_test_scaled)).clip(min=0)
sub_stacking = pd.DataFrame({'Id': test_ids, 'SalePrice': preds_stacking})
sub_stacking.to_csv(SUBMISSIONS_DIR / 'stacking_top3.csv', index=False)

print("Submissions saved:")
print(f"  catboost_fe.csv:   {sub_cat['SalePrice'].mean():,.0f} средняя цена")
print(f"  voting_top3.csv:   {sub_voting['SalePrice'].mean():,.0f} средняя цена")
print(f"  stacking_top3.csv: {sub_stacking['SalePrice'].mean():,.0f} средняя цена")

Submissions saved:
  catboost_fe.csv:   178,324 средняя цена
  voting_top3.csv:   178,199 средняя цена
  stacking_top3.csv: 178,982 средняя цена
